# LEP-21 — Construire un modèle de référence

## Objectif

Estimer l’espérance de vie à la naissance d’un pays à partir de ses indicateurs de la même année. Nous utilisons une régression linéaire multiple avec les 18 variables d’entrée retenues dans LEP-20.

## Protocole d’évaluation

Le modèle est entraîné sur les données de **2000 à 2011** et évalué sur celles de **2012 à 2013**. Les valeurs numériques manquantes sont imputées à partir des médianes calculées sur l’ensemble d’entraînement. Les années **2014 et 2015** sont réservées au test final.

Nous mesurons les erreurs de prédiction à l’aide de la **MAE** et de la **RMSE**, toutes deux exprimées en années. Ces résultats serviront de référence pour comparer les modèles dans **LEP-22**.

## 1. Chargement du jeu de modélisation et modélisation

In [2]:
# Bibliothèques
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

from pretraitement_lep import imputer_par_statut


In [3]:
# Charger le jeu de modélisation et reprendre le découpage de LEP-20.
fichier = Path("../data/processed/life_expectancy_modelisation.csv")
df = pd.read_csv(fichier)

cible = "life_expectancy"
colonnes_exclues = {"country", "iso3", "year", "ensemble", cible}
variables_entree = [colonne for colonne in df.columns if colonne not in colonnes_exclues]
variables_numeriques = [colonne for colonne in variables_entree if colonne != "status"]

train = df.loc[df["ensemble"].eq("entrainement")]
val = df.loc[df["ensemble"].eq("validation")]

X_train, y_train = train[variables_entree], train[cible]
X_val, y_val = val[variables_entree], val[cible]

# Imputer les deux ensembles avec les médianes de l'entraînement.
X_train_impute = imputer_par_statut(X_train, X_train, variables_numeriques)
X_val_impute = imputer_par_statut(X_train, X_val, variables_numeriques)

# Encoder le statut de développement et conserver les indicateurs numériques.
encodage = ColumnTransformer(
    [("statut", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), ["status"])],
    remainder="passthrough",
)
modele_reference = make_pipeline(encodage, LinearRegression())

# Entraîner le modèle, puis mesurer ses erreurs sur la validation.
modele_reference.fit(X_train_impute, y_train)
predictions_val = modele_reference.predict(X_val_impute)

mae = mean_absolute_error(y_val, predictions_val)
rmse = mean_squared_error(y_val, predictions_val) ** 0.5

print(f"Variables d'entrée : {len(variables_entree)}")
print(f"Observations d'entraînement : {len(y_train)}")
print(f"Observations de validation : {len(y_val)}")
print(f"MAE de validation : {mae:.2f} ans")
print(f"RMSE de validation : {rmse:.2f} ans")

Variables d'entrée : 18
Observations d'entraînement : 2196
Observations de validation : 374
MAE de validation : 1.95 ans
RMSE de validation : 2.58 ans


## 2.Analyse des erreurs par année de validation

In [4]:
erreurs_val = val[["year", "country", "life_expectancy"]].copy()
erreurs_val["prediction"] = predictions_val
erreurs_val["erreur"] = erreurs_val["prediction"] - erreurs_val["life_expectancy"]
erreurs_val["erreur_absolue"] = erreurs_val["erreur"].abs()

bilan_annuel = erreurs_val.groupby("year").agg(
    observations=("erreur", "size"),
    mae=("erreur_absolue", "mean"),
    erreur_moyenne=("erreur", "mean"),
)

display(bilan_annuel.round(2))

,observations,mae,erreur_moyenne
year,,,
2012,183,1.89,0.35
2013,191,2.01,0.45


Conclusion de LEP-21
La régression linéaire multiple obtient une MAE de 1,95 an et une RMSE de 2,58 ans sur les 374 observations de validation. La MAE est proche d’une année à l’autre, avec 1,89 an en 2012 et 2,01 ans en 2013.

Le modèle tend à surestimer légèrement l’espérance de vie sur les deux années. L’erreur moyenne signée est de +0,35 an en 2012 et de +0,45 an en 2013.

Cette régression servira de référence pour comparer les modèles de LEP-22 sur les mêmes observations de validation. Les années 2014 et 2015 restent réservées au test final.

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer

A supprimer